In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Complete_DataFrame_Transformations_Actions")
    .master("local[*]")   
    .getOrCreate()
)

spark

In [2]:
employee_data = [
    (101, "Anuj",   "IT",      90000, 29, "Mumbai",    1001, "2023-01-10", 4.7, None),
    (102, "Riya",   "HR",      60000, 31, "Pune",      1002, "2022-11-05", 4.2, "A"),
    (103, "Vikas",  "IT",      85000, 27, "Bengaluru", 1001, "2024-02-12", 4.5, "B"),
    (104, "Sneha",  "Finance", 95000, 35, "Mumbai",    1003, "2021-08-19", 4.8, "A"),
    (105, "Amit",   "Sales",   50000, 26, "Delhi",     1004, "2023-06-01", 3.9, None),
    (106, "Pooja",  "HR",      65000, 30, "Chennai",   1002, "2020-12-11", 4.1, "B"),
    (107, "Karan",  "IT",     120000, 33, "Hyderabad", 1001, "2019-03-14", 4.9, "A"),
    (108, "Meera",  "Sales",   52000, 28, "Pune",      1004, "2024-01-18", 3.7, "C"),
    (109, "Rohit",  "Finance", 88000, 32, "Delhi",     1003, "2022-09-09", 4.0, "B"),
    (110, "Nisha",  "Ops",     70000, 29, "Mumbai",    1005, "2023-04-22", 4.3, None),
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), False),
    StructField("dept", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("grade", StringType(), True),
])

emp_df = spark.createDataFrame(employee_data, employee_schema) \
    .withColumn("join_date", F.to_date("join_date"))

In [3]:
emp_df.show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|   Nisha

In [4]:
import pyspark.sql.functions as F

In [6]:
emp_df.count()

10

In [11]:
emp_df.groupBy("dept").agg(F.count("*").alias("numbers")).show()

+-------+-------+
|   dept|numbers|
+-------+-------+
|     HR|      2|
|     IT|      3|
|Finance|      2|
|  Sales|      2|
|    Ops|      1|
+-------+-------+



In [ ]:
emp_df.groupBy("dept").count().show()

In [12]:
emp_df.groupBy("dept").sum("salary").show()

+-------+-----------+
|   dept|sum(salary)|
+-------+-----------+
|     HR|     125000|
|     IT|     295000|
|Finance|     183000|
|  Sales|     102000|
|    Ops|      70000|
+-------+-----------+



In [14]:
emp_df.groupBy("dept").avg("salary").show()

+-------+-----------------+
|   dept|      avg(salary)|
+-------+-----------------+
|     HR|          62500.0|
|     IT|98333.33333333333|
|Finance|          91500.0|
|  Sales|          51000.0|
|    Ops|          70000.0|
+-------+-----------------+



In [18]:
emp_df.groupBy("dept").agg(
    F.max("salary"),
    F.count("dept"),
    F.sum("salary"),
    F.avg("salary")
).show()

+-------+-----------+-----------+-----------+-----------------+
|   dept|max(salary)|count(dept)|sum(salary)|      avg(salary)|
+-------+-----------+-----------+-----------+-----------------+
|     HR|      65000|          2|     125000|          62500.0|
|     IT|     120000|          3|     295000|98333.33333333333|
|Finance|      95000|          2|     183000|          91500.0|
|  Sales|      52000|          2|     102000|          51000.0|
|    Ops|      70000|          1|      70000|          70000.0|
+-------+-----------+-----------+-----------+-----------------+



In [19]:
emp_df.groupBy("dept").agg(
    F.avg("salary").alias("avg_salary")
).filter(F.col("avg_salary")>80000).show()

+-------+-----------------+
|   dept|       avg_salary|
+-------+-----------------+
|     IT|98333.33333333333|
|Finance|          91500.0|
+-------+-----------------+

